# Notebook 2: Groundedness grading with Azure OpenAI

This notebook loads the synthetic groundedness QnA dataset built in Notebook 1 and evaluates multiple Azure OpenAI deployments as graders using three prompt templates. It also compares the model-based graders against Azure AI Evaluation's `GroundednessEvaluator`, explores stochasticity via multiple runs, and is ready to log results to an Azure AI Foundry project.


In [ ]:
import os
import json
from typing import List, Dict, Callable

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
from openai import OpenAI
from azure.ai.evaluation import GroundednessEvaluator, evaluate

# Azure OpenAI configuration (environment variables should already be set)
client = OpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT").rstrip("/") + "/openai/v1/",
)

MODEL_DEPLOYMENTS = [
    {"name": "gpt-4o-mini-eval", "deployment": "gpt-4o-mini-eval"},
    {"name": "gpt-4o-eval", "deployment": "gpt-4o-eval"},
]

AZURE_AI_PROJECT_URL = "<paste your Azure AI Foundry project URL here>"

N_RUNS_PER_QUESTION = 5  # number of stochastic runs per question / template / model
LLM_TEMPERATURE = 0.3    # non-zero to expose stochasticity


In [ ]:
DATA_PATH = "synthetic_groundedness_qna.jsonl"
with open(DATA_PATH, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

df = pd.DataFrame(records)


def classification_metrics(y_true, y_pred):
    """
    Compute simple metrics for ordinal 1-5 predictions:
    - exact label match
    - within-1 accuracy
    - Pearson correlation
    - Spearman correlation
    """
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    exact = (y_true == y_pred).mean()
    within_1 = (np.abs(y_true - y_pred) <= 1).mean()
    pearson = pearsonr(y_true, y_pred)[0]
    spearman = spearmanr(y_true, y_pred)[0]
    return {
        "exact_match": float(exact),
        "within_1": float(within_1),
        "pearson": float(pearson),
        "spearman": float(spearman),
    }

# Uncomment to inspect the dataset distribution
# display(df.head())
# display(df["ground_truth_score"].value_counts())


In [ ]:
RUBRIC_FOR_PROMPTS_NUMERIC = {
    1: "Severely ungrounded or contradictory - mostly unrelated to the context or clearly conflicts with it.",
    2: "Largely ungrounded - only superficial overlap with the context; most claims can't be verified from the context.",
    3: "Mixed - some important claims are supported by the context, others are not clearly supported.",
    4: "Mostly grounded - the main answer is supported by the context, with only minor extra elaboration.",
    5: "Fully grounded - all important claims can be directly supported by the context; no speculative additions.",
}

RUBRIC_FOR_PROMPTS_ALPHA = {
    "E": "Severely ungrounded or contradictory.",
    "D": "Largely ungrounded - only superficial overlap with the context; most claims are unsupported.",
    "C": "Mixed - some important claims are supported, others are not clearly supported.",
    "B": "Mostly grounded - the main answer is supported by the context, with only minor extra elaboration.",
    "A": "Fully grounded - all important claims are clearly supported by the context.",
}

LABEL_TO_SCORE = {"A": 5, "B": 4, "C": 3, "D": 2, "E": 1}


In [ ]:
def build_system_and_user_messages(template_id: str, row: Dict):
    question = row["question"]
    context = row["context"]
    answer = row["answer"]

    if template_id == "numeric_free":
        system_text = (
            "You are an expert evaluator of answer groundedness.
"
            "Given a question, a retrieved context, and an answer, rate how well the answer is grounded "
            "in the context on a scale from 1 (not grounded) to 5 (fully grounded).
"
            "Grounded answers base their claims on the context; ungrounded answers invent facts or contradict the context.
"
            "Only output the integer 1, 2, 3, 4, or 5 with no extra text."
        )
    elif template_id == "numeric_rubric":
        system_text = (
            "You are an expert judge of how well answers are grounded in a given context.
"
            "Use the following 1-5 groundedness scale:
"
            "1: Severely ungrounded or contradictory - mostly unrelated to the context or clearly conflicts with it.
"
            "2: Largely ungrounded - only superficial overlap with the context; most claims can't be verified from the context.
"
            "3: Mixed - some important claims are supported by the context, others are not clearly supported.
"
            "4: Mostly grounded - the main answer is supported by the context, with only minor extra elaboration.
"
            "5: Fully grounded - all important claims can be directly supported by the context; no speculative additions.
"
            "Return only a single integer 1, 2, 3, 4, or 5."
        )
    elif template_id == "alpha_rubric":
        system_text = (
            "You are an expert judge of how well answers are grounded in a given context.
"
            "Use the following labels:
"
            "A: Fully grounded - all important claims are clearly supported by the context.
"
            "B: Mostly grounded - the main answer is supported by the context, with only minor extra elaboration.
"
            "C: Mixed - some important claims are supported, others are not clearly supported.
"
            "D: Largely ungrounded - only superficial overlap with the context; most claims are unsupported.
"
            "E: Severely ungrounded or contradictory - content is unrelated to or in conflict with the context.
"
            "Choose exactly one label (A, B, C, D, or E) and output just that single letter."
        )
    else:
        raise ValueError(f"Unknown template_id: {template_id}")

    if template_id == "alpha_rubric":
        final_prompt_line = "Groundedness label (A-E):"
    else:
        final_prompt_line = "Groundedness score (1-5):"

    user_text = f"""Question:
{question}

Retrieved context:
{context}

Answer:
{answer}

{final_prompt_line}
"""
    return system_text, user_text


def grade_once_with_template(
    deployment: str,
    template_id: str,
    row: Dict,
) -> float:
    """Call a given Azure OpenAI deployment with a grading template and return a numeric score 1-5."""
    system_text, user_text = build_system_and_user_messages(template_id, row)

    response = client.chat.completions.create(
        model=deployment,  # Azure: model=deployment_name
        messages=[
            {"role": "system", "content": system_text},
            {"role": "user", "content": user_text},
        ],
        temperature=LLM_TEMPERATURE,
        max_tokens=4,
    )

    raw = response.choices[0].message.content.strip()

    if template_id == "alpha_rubric":
        label = raw[0].upper() if raw else ""
        score = LABEL_TO_SCORE.get(label, np.nan)
    else:
        digits = "".join(ch for ch in raw if ch.isdigit())
        score = float(digits) if digits in {"1", "2", "3", "4", "5"} else np.nan

    return score


In [ ]:
all_preds = []

templates = ["numeric_free", "numeric_rubric", "alpha_rubric"]

for model in MODEL_DEPLOYMENTS:
    for template_id in templates:
        for run_idx in range(N_RUNS_PER_QUESTION):
            print(f"Running model={model['name']}, template={template_id}, run={run_idx}")
            for _, row in df.iterrows():
                pred_score = grade_once_with_template(
                    deployment=model["deployment"],
                    template_id=template_id,
                    row=row,
                )
                all_preds.append({
                    "id": row["id"],
                    "ground_truth_score": row["ground_truth_score"],
                    "model": model["name"],
                    "deployment": model["deployment"],
                    "template": template_id,
                    "run": run_idx,
                    "pred_score": pred_score,
                })

preds_df = pd.DataFrame(all_preds)
# preds_df.head()


In [ ]:
results = []

for (model_name, template_id, run_idx), group in preds_df.groupby(["model", "template", "run"]):
    metrics = classification_metrics(
        group["ground_truth_score"],
        group["pred_score"],
    )
    metrics.update({
        "model": model_name,
        "template": template_id,
        "run": run_idx,
    })
    results.append(metrics)

metrics_df = pd.DataFrame(results)
# metrics_df.head()

agg_metrics = (
    metrics_df
    .groupby(["model", "template"])
    .agg(["mean", "std"])
)
# agg_metrics


In [ ]:
sns.set(style="whitegrid")


def plot_kdes_for_model_template(model_name: str, template_id: str, preds: pd.DataFrame):
    """For a given model and template, plot KDEs over predicted scores for each true groundedness level (1-5)."""
    subset = preds[(preds["model"] == model_name) & (preds["template"] == template_id)]
    fig, axes = plt.subplots(1, 5, figsize=(20, 3), sharey=True)

    for g, ax in zip(range(1, 6), axes):
        data_g = subset[subset["ground_truth_score"] == g]["pred_score"].dropna()
        if len(data_g) > 1:
            sns.kdeplot(data_g, ax=ax)
        else:
            ax.hist(data_g, bins=np.linspace(0.5, 5.5, 6), density=True)

        ax.axvline(g, linestyle="--", linewidth=1)
        ax.set_title(f"True level = {g}")
        ax.set_xlim(1, 5)
        ax.set_xlabel("Predicted score")

    fig.suptitle(f"Groundedness predictions for model={model_name}, template={template_id}")
    plt.tight_layout()
    plt.show()


for model in MODEL_DEPLOYMENTS:
    for template_id in ["numeric_free", "numeric_rubric", "alpha_rubric"]:
        plot_kdes_for_model_template(model["name"], template_id, preds_df)


In [ ]:
model_config = {
    "azure_endpoint": os.environ.get("AZURE_OPENAI_ENDPOINT"),
    "api_key": os.environ.get("AZURE_OPENAI_API_KEY"),
    "azure_deployment": MODEL_DEPLOYMENTS[0]["deployment"],
}

groundedness_eval = GroundednessEvaluator(model_config=model_config)

sdk_scores = []
for _, row in df.iterrows():
    result = groundedness_eval(
        response=row["answer"],
        context=row["context"],
        query=row["question"],
    )
    score = float(result["groundedness"])
    sdk_scores.append(score)

df["pred_sdk_groundedness"] = sdk_scores

sdk_metrics = classification_metrics(df["ground_truth_score"], df["pred_sdk_groundedness"])
sdk_metrics["model"] = "AzureAI_GroundednessEvaluator"
sdk_metrics["template"] = "built_in"
sdk_metrics["run"] = None

metrics_df = pd.concat([metrics_df, pd.DataFrame([sdk_metrics])], ignore_index=True)
# metrics_df.tail()


In [ ]:
eval_output = evaluate(
    data=DATA_PATH,
    evaluators={"groundedness": groundedness_eval},
    azure_ai_project=AZURE_AI_PROJECT_URL,
    evaluator_config={
        "groundedness": {
            "column_mapping": {
                "response": "${data.answer}",
                "context": "${data.context}",
                "query": "${data.question}",
            }
        }
    },
    output_path="azure_groundedness_eval_results.json",
)

print(eval_output.get("metrics", {}))
